# 01 — LangGraph
### Stateful Agents · Graphs · Conditional Routing · Human-in-the-Loop · Multi-Agent



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1aGM9qe80BI5MYVafpe1NXlHwQhhgAlrs)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://python.org)
[![LangChain](https://img.shields.io/badge/LangChain-0.3.x-green)](https://python.langchain.com)
[![Gemini](https://img.shields.io/badge/LLM-Gemini%201.5%20Flash-orange)](https://aistudio.google.com)


### Copy the links from below badge to visit my pages


[![GitHub](https://img.shields.io/badge/GitHub-View_Profile-black?logo=github)](https://github.com/mtptisid)
<a href="https://www.linkedin.com/in/siddharamayya-mathapati" target="_blank">
  <img src="https://img.shields.io/badge/LinkedIn-Connect-blue?logo=linkedin" />
</a>
[![Portfolio](https://img.shields.io/badge/Portfolio-Visit-orange?logo=google-chrome)](https://siddharamayya.in)



> **Part 1 of the LangChain Tutorial Series.**
> LangChain chains are linear. LangGraph makes your agents stateful, cyclical,
> and self-correcting — with full control over branching logic, memory across
> sessions, and the ability to pause and wait for human input.


## What You Will Build

By the end of this notebook you will have built:

1. A simple two-node graph (your first LangGraph app)
2. A ReAct agent with tools — rebuilt properly as a graph
3. A self-correcting agent that checks its own answers and retries if unsure
4. A human-in-the-loop workflow that pauses and waits for your approval
5. A multi-agent graph where a supervisor routes tasks to specialist agents

---

## Why LangGraph? What LangChain Chains Cannot Do

In [LangChain](https://github.com/mtptisid/Langchain) you built chains like this:

```
prompt | llm | output_parser
```

That's a **DAG** — a Directed Acyclic Graph. It flows in one direction,
one time, and stops. It cannot:

- Loop back and retry when the output is wrong
- Branch differently based on what the LLM decides
- Pause mid-execution and wait for a human to approve something
- Maintain memory across separate conversation sessions
- Run multiple specialized agents that hand off tasks to each other

LangGraph solves all of this by letting you define your agent as an actual
**graph** — with nodes (steps), edges (connections), and — crucially — **cycles**
(loops back to earlier steps).

```
LangChain chain:     A → B → C → END          (linear, one pass)
LangGraph graph:     A → B → C → A (retry)    (cyclic, stateful)
                               ↓
                              END
```

---

# The Mental Model

Everything in LangGraph is built from three things:

```
┌──────────────────────────────────────────────────────────┐
│  State   → a shared dict that every node can read/write  │
│  Nodes   → Python functions that transform the state     │
│  Edges   → connections between nodes (fixed or dynamic)  │
└──────────────────────────────────────────────────────────┘
```

The graph runs by:
1. Starting with an initial state
2. Passing the state through nodes one at a time
3. Using edges to decide which node comes next
4. Stopping when it reaches `END`

Every node receives the current state and returns updates to it.
The state accumulates information as it flows through the graph.

---



## Setup


In [2]:
!pip install -U -q langgraph langchain langchain-community \
            langchain-google-genai langchain-text-splitters \
            google-generativeai wikipedia duckduckgo-search

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 whi

In [5]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_KEYS")

## Part 1 — Your First Graph (Two Nodes)

Before agents, understand the structure with the simplest possible graph:
one node that calls an LLM, one node that formats the output.

In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# ── Step 1: Define State ───────────────────────────────────────────────────
# State is a TypedDict — a plain Python dict with typed keys.
# Every node reads from this and writes updates back to it.

class SimpleState(TypedDict):
    question: str    # input question
    answer:   str    # LLM's answer
    summary:  str    # formatted final output

# ── Step 2: Define Nodes ───────────────────────────────────────────────────
# Each node is a function: (state) → dict of updates

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

def call_llm(state: SimpleState) -> dict:
    """Node 1: Send the question to the LLM."""
    response = llm.invoke([HumanMessage(content=state["question"])])
    return {"answer": response.content}

def format_output(state: SimpleState) -> dict:
    """Node 2: Format the answer into a final summary."""
    summary = f"Q: {state['question']}\n\nA: {state['answer']}"
    return {"summary": summary}

# ── Step 3: Build the Graph ────────────────────────────────────────────────
builder = StateGraph(SimpleState)

# Add nodes (name → function)
builder.add_node("llm_call", call_llm)
builder.add_node("format",   format_output)

# Add edges (flow between nodes)
builder.set_entry_point("llm_call")          # start here
builder.add_edge("llm_call", "format")       # then go here
builder.add_edge("format", END)              # then stop

# Compile into a runnable
graph = builder.compile()

# ── Step 4: Run ────────────────────────────────────────────────────────────
result = graph.invoke({"question": "What is LangGraph in one sentence?"})
print(result["summary"])

Q: What is LangGraph in one sentence?

A: LangGraph is a library for building stateful, multi-actor LLM applications and agents by modeling their logic as a graph with explicit support for cycles.


**State after each node:**

```
Initial:  {"question": "What is LangGraph?", "answer": "", "summary": ""}
After node 1 (call_llm):   {"question": "...", "answer": "LangGraph is...", "summary": ""}
After node 2 (format):     {"question": "...", "answer": "...", "summary": "Q: ...\nA: ..."}
```

Nodes only need to return the keys they update — unchanged keys are preserved automatically.

---


## What's Next

**[→ Notebook 02: Conditional Edges (Branching)](./02_edges_in_langgraph.ipynb)**
Fixed edges always go to the same next node. **Conditional edges** let the
graph branch differently based on what's in the state — this is how agents
decide whether to keep working or stop.

---

*Part of the [LangGraphTutorial Series](../README.md) — built with LangChain 0.3.x and Google Gemini 1.5 Flash*